In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

class HeteroMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=83, n_layers=7, dropout=0.0):
        super().__init__()
        layers = []
        layers.append(nn.Linear(in_dim, hidden))
        layers.append(nn.Tanh())
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(hidden, hidden))
            layers.append(nn.Tanh())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        layers.append(nn.Linear(hidden, 2*out_dim)) 
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        out = self.net(x)
        mu, log_var = out.chunk(2, dim=1)
        mu = (torch.tanh(mu) + 1) / 2
        log_var = torch.clamp(log_var, -5, 5)
        var = torch.exp(log_var) + 1e-8
        return mu, var, log_var
    
def nll_loss(mu, log_var, y):
    var = torch.exp(log_var)
    return 0.5 * torch.mean((y - mu)**2 / var + log_var)

model = HeteroMLP(in_dim=41, out_dim=1)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
best_val = float('inf')
patience = 50
counter = 0
max_epochs = 1000

x_train = np.load('xlo_oversampling.npy')[:,:]
y_train = np.load('ylo_oversampling.npy')[:,:]
x_val   = np.load('xlo_val.npy')[:,:]
y_val   = np.load('ylo_val.npy')[:,:]
y_val[:,1]=y_val[:,1]+y_val[:,3]
y_val=np.copy(y_val[:,[1]])

x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1)

x_val = torch.tensor(x_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32).reshape(-1,1)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=512, shuffle=True)
val_loader   = DataLoader(TensorDataset(x_val, y_val), batch_size=512)
ep1=np.zeros([0,2])
for epoch in range(max_epochs):
    print('epoch=',epoch)

    # ---- TRAIN ----
    model.train()
    train_loss = 0
    for xb, yb in train_loader:
        mu, var, log_var = model(xb)

        loss = nll_loss(mu, log_var, yb)

        opt.zero_grad()
        loss.backward()
        opt.step()

        train_loss += loss.item()
    PATH = f"checkpoint/hetero_mlp_epoch_{epoch}.pt"
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
    }, PATH)

    # ---- VALIDATION ----
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            mu, var, log_var = model(xb)

            loss = nll_loss(mu, log_var, yb)
            val_loss += loss.item()

    val_loss /= len(val_loader)
    ep=np.reshape([epoch,val_loss],(1,2))
    ep1=np.append(ep1,ep,axis=0)

    # ---- EARLY STOPPING ----
    if val_loss < best_val:
        best_val = val_loss
        best_model = model.state_dict()
        counter = 0
    else:
        counter += 1
        if counter > patience:
            break

model.load_state_dict(best_model)
model.eval()
torch.save({
    'model_state_dict': best_model,
    'optimizer_state_dict': opt.state_dict(),
    'best_val_loss': best_val
}, "best_hetero_mlp.pt")
with open('loss.npy','wb') as f:
    np.save(f,ep1)

# model = HeteroMLP(in_dim=41, out_dim=1, hidden=83, n_layers=7)
# checkpoint = torch.load("best_hetero_mlp.pt")
# model.load_state_dict(checkpoint['model_state_dict'])
# opt = torch.optim.Adam(model.parameters(), lr=1e-3)
# opt.load_state_dict(checkpoint['optimizer_state_dict'])
# model.eval()

# xtest=np.load('xlo_test.npy')
# ytest=np.load('ylo_test.npy')
# ytest[:,1]=ytest[:,1]+ytest[:,3]
# ytest=np.copy(ytest[:,[1]])
# xtest=torch.tensor(xtest, dtype=torch.float32)
# ytest=torch.tensor(ytest, dtype=torch.float32).reshape(-1,1)
# model.eval()
# mu, var, log_var = model(xtest)